# GCS Bucket Data Management

Replace old system documents on the GCS bucket with new data from `data/` folder.

**GCS Path Structure (Option A — Structured):**
```
system/{grade}/{doc_type}/{subject}/{filename}
```

**Steps:**
1. Load config from `.env` via `get_settings()`
2. List current bucket contents
3. Delete all files under `system/`
4. Build upload manifest from local `data/` folder
5. Upload all PDFs with metadata (`user_id`, `scope`, `grade`, `subject`, `doc_type`)
6. Verify upload
7. Trigger Vertex AI Search FULL re-import


In [3]:
# Cell 1: Imports & Config (loaded from .env)
import sys
from pathlib import Path

# Ensure project root is on sys.path for src imports
PROJECT_ROOT = Path.cwd().parent.parent  # notebooks/exploration/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from google.cloud import storage
from google.cloud import discoveryengine_v1 as discoveryengine
import time

from src.config import get_settings
from src.config.constants import SYSTEM_USER_ID

settings = get_settings()

PROJECT_ID = settings.gcp_project_id
BUCKET_NAME = settings.gcs_bucket
DATA_STORE_ID = settings.data_store_id
DATA_STORE_LOCATION = settings.data_store_location
DATA_DIR = PROJECT_ROOT / "data"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

print(f"Project: {PROJECT_ID}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Data Store: {DATA_STORE_ID} ({DATA_STORE_LOCATION})")
print(f"Data dir: {DATA_DIR} (exists: {DATA_DIR.exists()})")


Project: green-mercury-485016-n1
Bucket: documents-development-bucket
Data Store: aiservice-datastore-m1_1772802306291 (global)
Data dir: /home/sakana/Code/ManifoldsTeam/AIServices/data (exists: True)


In [4]:
# Cell 2: List current bucket contents under system/
system_blobs = list(bucket.list_blobs(prefix="system/"))
system_files = [b for b in system_blobs if not b.name.endswith("/")]

print(f"Total blobs under system/: {len(system_blobs)}")
print(f"Total files under system/: {len(system_files)}")
print("\nSample paths:")
for b in system_files[:10]:
    print(f"  {b.name}  ({b.size} bytes)")
if len(system_files) > 10:
    print(f"  ... and {len(system_files) - 10} more")


Total blobs under system/: 227
Total files under system/: 226

Sample paths:
  system/2026-03-07/test/math_grade10_system.txt  (583 bytes)
  system/sgk/lop-10/cong-nghe/sach-giao-khoa-cong-nghe-10.pdf  (12127066 bytes)
  system/sgk/lop-10/dia-li/chuyen-de-hoc-tap-dia-li-10-chan-troi-sang-tao.pdf  (3229586 bytes)
  system/sgk/lop-10/dia-li/chuyen-de-hoc-tap-dia-li-10-ket-noi-tri-thuc-voi-cuoc-song.pdf  (6880629 bytes)
  system/sgk/lop-10/dia-li/sach-giao-khoa-dia-li-10-chan-troi-sang-tao.pdf  (15279117 bytes)
  system/sgk/lop-10/dia-li/sach-giao-khoa-dia-li-10-nang-cao.pdf  (21697670 bytes)
  system/sgk/lop-10/dia-li/sach-giao-khoa-dia-li-10.pdf  (30414719 bytes)
  system/sgk/lop-10/gdcd/chuyen-de-hoc-tap-gdktpl-10-chan-troi-sang-tao.pdf  (7215671 bytes)
  system/sgk/lop-10/gdcd/chuyen-de-hoc-tap-gdktpl-10-ket-noi-tri-thuc-voi-cuoc-song.pdf  (15497035 bytes)
  system/sgk/lop-10/gdcd/sach-giao-khoa-gdktpl-10-chan-troi-sang-tao.pdf  (16786792 bytes)
  ... and 216 more


In [5]:
# Cell 3: Delete ALL files under system/
print(f"Deleting {len(system_files)} files under system/ ...")

BATCH_SIZE = 100
deleted = 0
for i in range(0, len(system_files), BATCH_SIZE):
    batch = system_files[i:i + BATCH_SIZE]
    with client.batch():
        for blob in batch:
            blob.delete()
    deleted += len(batch)
    print(f"  Deleted {deleted}/{len(system_files)}")

print(f"\nDone. Deleted {deleted} files from system/")


Deleting 226 files under system/ ...
  Deleted 100/226
  Deleted 200/226
  Deleted 226/226

Done. Deleted 226 files from system/


In [6]:
# Cell 4: Verify deletion
remaining = list(bucket.list_blobs(prefix="system/"))
remaining_files = [b for b in remaining if not b.name.endswith("/")]
print(f"Remaining files under system/: {len(remaining_files)}")
if remaining_files:
    for b in remaining_files[:5]:
        print(f"  STILL EXISTS: {b.name}")


Remaining files under system/: 0


In [7]:
# Cell 5: Collect local files and build upload manifest
local_pdfs = sorted(DATA_DIR.rglob("*.pdf"))
print(f"Total local PDFs: {len(local_pdfs)}")

upload_manifest = []
for pdf_path in local_pdfs:
    # Pattern: data/{grade}/{doc_type}/{subject}/{filename}.pdf
    rel = pdf_path.relative_to(DATA_DIR)
    parts = rel.parts
    if len(parts) < 4:
        print(f"  SKIP (unexpected path): {rel}")
        continue
    grade = parts[0]       # lop-10
    doc_type = parts[1]    # sgk or sbt
    subject = parts[2]     # toan, hoa-hoc, etc.
    filename = parts[3]    # actual filename

    # GCS path: system/{grade}/{doc_type}/{subject}/{filename}
    gcs_path = f"system/{grade}/{doc_type}/{subject}/{filename}"

    upload_manifest.append({
        "local_path": pdf_path,
        "gcs_path": gcs_path,
        "grade": grade,
        "doc_type": doc_type,
        "subject": subject,
        "filename": filename,
    })

print(f"Upload manifest: {len(upload_manifest)} files")
print("\nBreakdown by grade/type:")
from collections import Counter
for key, count in sorted(Counter((m['grade'], m['doc_type']) for m in upload_manifest).items()):
    print(f"  {key[0]}/{key[1]}: {count} files")


Total local PDFs: 798
Upload manifest: 798 files

Breakdown by grade/type:
  lop-10/sbt: 326 files
  lop-10/sgk: 48 files
  lop-11/sbt: 165 files
  lop-11/sgk: 73 files
  lop-12/sbt: 124 files
  lop-12/sgk: 62 files


In [8]:
# Cell 6: Upload all files with metadata
import concurrent.futures

def upload_one(item: dict) -> str:
    """Upload a single file to GCS with metadata."""
    blob = bucket.blob(item["gcs_path"])
    blob.metadata = {
        "user_id": SYSTEM_USER_ID,
        "scope": "system",
        "grade": item["grade"],
        "subject": item["subject"],
        "doc_type": item["doc_type"],
    }
    blob.upload_from_filename(str(item["local_path"]), content_type="application/pdf")
    return item["gcs_path"]

print(f"Uploading {len(upload_manifest)} files to gs://{BUCKET_NAME}/system/ ...")
start = time.time()

uploaded = 0
errors = []

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(upload_one, item): item for item in upload_manifest}
    for future in concurrent.futures.as_completed(futures):
        item = futures[future]
        try:
            future.result()
            uploaded += 1
            if uploaded % 50 == 0:
                print(f"  Uploaded {uploaded}/{len(upload_manifest)}")
        except Exception as e:
            errors.append((item["gcs_path"], str(e)))

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s. Uploaded: {uploaded}, Errors: {len(errors)}")
if errors:
    for path, err in errors[:5]:
        print(f"  ERROR: {path} -> {err}")


Uploading 798 files to gs://documents-development-bucket/system/ ...
  Uploaded 50/798
  Uploaded 100/798
  Uploaded 150/798
  Uploaded 200/798
  Uploaded 250/798
  Uploaded 300/798
  Uploaded 350/798
  Uploaded 400/798
  Uploaded 450/798
  Uploaded 500/798
  Uploaded 550/798
  Uploaded 600/798
  Uploaded 650/798
  Uploaded 700/798
  Uploaded 750/798

Done in 703.3s. Uploaded: 795, Errors: 3
  ERROR: system/lop-11/sgk/gdcd/sach-giao-khoa-gdktpl-11-canh-dieu.pdf -> Timeout of 120.0s exceeded, last exception: ('Connection aborted.', TimeoutError('The write operation timed out'))
  ERROR: system/lop-11/sgk/lich-su/sach-giao-khoa-lich-su-11-nang-cao.pdf -> Timeout of 120.0s exceeded, last exception: ('Connection aborted.', TimeoutError('The write operation timed out'))
  ERROR: system/lop-12/sgk/hoa-hoc/lop-12_sgk_hoa-hoc_sgk-hoa-hoc-12-pdf-hoa-hoc-12-ket-noi-tri-thuc_1ax4efED.pdf -> Timeout of 120.0s exceeded, last exception: ('Connection aborted.', TimeoutError('The write operation timed

In [9]:
# Cell 7: Verify upload - count files on bucket
uploaded_blobs = list(bucket.list_blobs(prefix="system/"))
uploaded_files = [b for b in uploaded_blobs if not b.name.endswith("/")]
print(f"Files on bucket under system/: {len(uploaded_files)}")

# Check a sample file's metadata
if uploaded_files:
    sample = uploaded_files[0]
    sample.reload()
    print(f"\nSample: {sample.name}")
    print(f"  Size: {sample.size} bytes")
    print(f"  Content-Type: {sample.content_type}")
    print(f"  Metadata: {sample.metadata}")

# Breakdown by grade/doc_type
from collections import Counter
grade_type_counts = Counter()
for b in uploaded_files:
    parts = b.name.split("/")
    if len(parts) >= 4:
        grade_type_counts[(parts[1], parts[2])] += 1
print("\nBreakdown:")
for key, count in sorted(grade_type_counts.items()):
    print(f"  {key[0]}/{key[1]}: {count}")


Files on bucket under system/: 798

Sample: system/lop-10/sbt/dia-li/chuyen-de-hoc-tap-dia-li-10-chan-troi-sang-tao.pdf
  Size: 3229586 bytes
  Content-Type: application/pdf
  Metadata: {'grade': 'lop-10', 'user_id': '__system__', 'scope': 'system', 'doc_type': 'sbt', 'subject': 'dia-li'}

Breakdown:
  lop-10/sbt: 326
  lop-10/sgk: 48
  lop-11/sbt: 165
  lop-11/sgk: 73
  lop-12/sbt: 124
  lop-12/sgk: 62


In [10]:
# Cell 8: Trigger Vertex AI Search FULL re-import
parent = (
    f"projects/{PROJECT_ID}/"
    f"locations/{DATA_STORE_LOCATION}/"
    f"collections/default_collection/"
    f"dataStores/{DATA_STORE_ID}/"
    f"branches/default_branch"
)

gcs_uri = f"gs://{BUCKET_NAME}/system/**"

print(f"Triggering FULL import...")
print(f"  Parent: {parent}")
print(f"  GCS URI: {gcs_uri}")

ds_client = discoveryengine.DocumentServiceClient()

gcs_source = discoveryengine.GcsSource(
    input_uris=[gcs_uri],
    data_schema="content",
)

request = discoveryengine.ImportDocumentsRequest(
    parent=parent,
    gcs_source=gcs_source,
    reconciliation_mode=discoveryengine.ImportDocumentsRequest.ReconciliationMode.FULL,
)

operation = ds_client.import_documents(request=request)
print(f"  Operation: {operation.operation.name}")
print("\nImport started. This may take a few minutes.")
print("Check status: GCP Console -> Agent Builder -> Data Stores")


Triggering FULL import...
  Parent: projects/green-mercury-485016-n1/locations/global/collections/default_collection/dataStores/aiservice-datastore-m1_1772802306291/branches/default_branch
  GCS URI: gs://documents-development-bucket/system/**
  Operation: projects/711906617662/locations/global/collections/default_collection/dataStores/aiservice-datastore-m1_1772802306291/branches/0/operations/import-documents-11607082215680860780

Import started. This may take a few minutes.
Check status: GCP Console -> Agent Builder -> Data Stores
